In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime, timedelta
import warnings
import subprocess
import sys

warnings.filterwarnings('ignore')

# Instalar datasets de HuggingFace si es necesario
try:
    import datasets
except ImportError:
    print("Instalando datasets...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "-q"])
    import datasets

# Configurar opciones de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("Librerías importadas correctamente")
print(f"  - pandas {pd.__version__}")
print(f"  - numpy {np.__version__}")
print(f"  - datasets {datasets.__version__}")

Librerías importadas correctamente
  - pandas 2.2.3
  - numpy 2.1.0
  - datasets 4.8.5


In [2]:
# Definir rutas de datasets desde Hugging Face
# Los datos se cargarán directamente desde https://huggingface.co/datasets/hao-li/AIDev

HF_REPO = "hao-li/AIDev"
HF_DATASETS = {
    "pull_request": "all_pull_request",
    "pull_request_popular": "pull_request",
    "pull_request_human": "human_pull_request",
    "repository": "repository",
    "pr_task_type": "pr_task_type",
    "pr_task_type_human": "human_pr_task_type",

    # NUEVAS
    "pr_commit_details": "pr_commit_details",
    "pr_comments": "pr_comments",
    "pr_review_comments": "pr_review_comments",
    "pr_reviews": "pr_reviews",
    "pr_timeline": "pr_timeline",
    "user": "user",
}

def load_hf_dataset(repo: str, config: str, name: str = "") -> pd.DataFrame:
    """
    Cargar dataset desde Hugging Face con manejo de errores.
    
    Args:
        repo: Nombre del dataset en HF (ej: hao-li/AIDev)
        config: Nombre de la configuración/split del dataset
        name: Nombre descriptivo para la salida
    
    Returns:
        DataFrame o DataFrame vacío si hay error
    """
    try:
        print(f"Cargando {name or config:40}", end=" ", flush=True)
        from datasets import load_dataset
        
        # Cargar dataset
        dataset = load_dataset(repo, name=config)
        
        # Convertir a pandas (manejo de diferentes estructuras)
        if isinstance(dataset, dict):
            df = dataset["train"].to_pandas() if "train" in dataset else dataset[list(dataset.keys())[0]].to_pandas()
        else:
            df = dataset.to_pandas()
        
        print(f"OK ({df.shape[0]:,} filas × {df.shape[1]} columnas)")
        return df
    
    except Exception as e:
        print(f"Error: {str(e)[:80]}")
        return pd.DataFrame()


# Cargar todos los datasets
print("\n" + "="*80)
print("CARGANDO DATASETS DESDE HUGGING FACE")
print("="*80 + "\n")

loaded_datasets = {}
for key, config in HF_DATASETS.items():
    loaded_datasets[key] = load_hf_dataset(HF_REPO, config, key)

# Asignar a variables globales para facilitar acceso
# NOTA: Usar pull_request (todos los PRs AI) como base principal
pull_request = loaded_datasets["pull_request"]
pull_request_popular = loaded_datasets["pull_request_popular"]
pull_request_human = loaded_datasets["pull_request_human"]
repository = loaded_datasets["repository"]
pr_task_type = loaded_datasets["pr_task_type"]
pr_task_type_human = loaded_datasets["pr_task_type_human"]

# Variables vacías (no disponibles en HF, se ignorarán)
pr_commit_details = pd.DataFrame()
pr_comments = pd.DataFrame()
pr_review_comments = pd.DataFrame()
pr_reviews = pd.DataFrame()
pr_timeline = pd.DataFrame()
user = pd.DataFrame()

print("\nCarga de datos desde Hugging Face completada")


CARGANDO DATASETS DESDE HUGGING FACE

Cargando pull_request                             

README.md: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

OK (932,791 filas × 14 columnas)
Cargando pull_request_popular                     

Generating train split:   0%|          | 0/33596 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/325500 [00:00<?, ? examples/s]

OK (325,500 filas × 8 columnas)
Cargando user                                     

user.parquet:   0%|          | 0.00/68.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1796 [00:00<?, ? examples/s]

OK (1,796 filas × 5 columnas)

Carga de datos desde Hugging Face completada


In [3]:
# ============================================================
# CARGA DIRECTA DE TABLAS GRANDES (PARQUET)
# ============================================================

print("\nCargando tablas extendidas...\n")

BASE_PATH = "hf://datasets/hao-li/AIDev/"

try:
    pr_commit_details = pd.read_parquet(
        BASE_PATH + "pr_commit_details.parquet"
    )
    
    print(f"pr_commit_details: {pr_commit_details.shape}")

except Exception as e:
    print(f"Error pr_commit_details: {e}")

try:
    pr_reviews = pd.read_parquet(
        BASE_PATH + "pr_reviews.parquet"
    )
    
    print(f"pr_reviews: {pr_reviews.shape}")

except Exception as e:
    print(f"Error pr_reviews: {e}")

try:
    pr_comments = pd.read_parquet(
        BASE_PATH + "pr_comments.parquet"
    )
    
    print(f"pr_comments: {pr_comments.shape}")

except Exception as e:
    print(f"Error pr_comments: {e}")

try:
    pr_timeline = pd.read_parquet(
        BASE_PATH + "pr_timeline.parquet"
    )
    
    print(f"pr_timeline: {pr_timeline.shape}")

except Exception as e:
    print(f"Error pr_timeline: {e}")

try:
    user = pd.read_parquet(
        BASE_PATH + "user.parquet"
    )
    
    print(f"user: {user.shape}")

except Exception as e:
    print(f"Error user: {e}")


Cargando tablas extendidas...



pr_commit_details: (711923, 14)
pr_reviews: (28875, 7)
pr_comments: (39122, 7)
pr_timeline: (325500, 8)
user: (1796, 5)


In [4]:
pr_commit_details.head()


,sha,pr_id,author,committer,message,commit_stats_total,commit_stats_additions,commit_stats_deletions,filename,status,additions,deletions,changes,patch
0,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,.github/workflows/compile-llms-txt.yml,added,38.0,0.0,38.0,"@@ -0,0 +1,38 @@\n+name: Compile llms.txt\n+\n..."
1,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,docs/compile_llms_txt.py,added,47.0,0.0,47.0,"@@ -0,0 +1,47 @@\n+import os\n+from pathlib im..."
2,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,llms.txt,added,22923.0,0.0,22923.0,None
3,dbd1b5f129f7cffa5ce284d7255814c98bcc38a2,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Fix lint issues: remove unused variable and ap...,35,18,17,docs/compile_llms_txt.py,modified,18.0,17.0,35.0,"@@ -1,47 +1,48 @@\n import os\n from pathlib i..."
4,c2659cfdedf666c8f14753d71664563c2a932b23,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Update llms.txt to follow official standard wi...,23035,89,22946,docs/compile_llms_txt.py,modified,51.0,36.0,87.0,"@@ -3,45 +3,60 @@\n \n \n def compile_llms_txt..."


In [5]:
# ============================================================
# FILTRAR ARCHIVOS CI/CD
# ============================================================

CI_PATTERNS = [
    r"\.github/workflows/",
    r"\.gitlab-ci\.yml",
    r"azure-pipelines\.yml",
    r"\.circleci/",
    r"Jenkinsfile",
    r"\.travis\.yml"
]

pattern = "|".join(CI_PATTERNS)

ci_commits = pr_commit_details[
    pr_commit_details["filename"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    )
]

print("="*80)
print("COMMITS RELACIONADOS A CI/CD")
print("="*80)

print(f"\nTotal encontrados: {len(ci_commits):,}")

display(ci_commits.head(10))

COMMITS RELACIONADOS A CI/CD

Total encontrados: 7,860


,sha,pr_id,author,committer,message,commit_stats_total,commit_stats_additions,commit_stats_deletions,filename,status,additions,deletions,changes,patch
0,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,.github/workflows/compile-llms-txt.yml,added,38.0,0.0,38.0,"@@ -0,0 +1,38 @@\n+name: Compile llms.txt\n+\n..."
7,b3c33fe3603254485dbf9f82b19d43ace616fc0a,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Enhance llms.txt with comprehensive repository...,4614,4520,94,.github/workflows/compile-llms-txt.yml,modified,4.0,0.0,4.0,"@@ -24,6 +24,10 @@ jobs:\n with:\n ..."
136,5ff2eeb80bbe498f9e143b95e0422c587b3dfa41,3126249131,devin-ai-integration[bot],devin-ai-integration[bot],feat(ci): add manifest-only Dockerfile testing...,69,69,0,.github/workflows/docker-connector-base-image-...,modified,69.0,0.0,69.0,"@@ -11,6 +11,7 @@ on:\n - docker-images/..."
138,b3d6d6b87ef5d863f7e46c95ab43e74e65992cf7,3126249131,devin-ai-integration[bot],devin-ai-integration[bot],fix(ci): adjust vulnerability scan severity fo...,2,1,1,.github/workflows/docker-connector-base-image-...,modified,1.0,1.0,2.0,"@@ -366,5 +366,5 @@ jobs:\n with:\n ..."
139,c4233d1aa8a118eaf58cc2ad271cb58dd20c29b0,3126249131,devin-ai-integration[bot],devin-ai-integration[bot],fix(ci): set fail-build false for manifest-onl...,3,2,1,.github/workflows/docker-connector-base-image-...,modified,2.0,1.0,3.0,"@@ -367,4 +367,5 @@ jobs:\n image: ""..."
143,65ac1c5318faaa21f3e6b1e5e9b172882cc75038,3126249131,aaronsteers,web-flow,Update .github/workflows/docker-connector-base...,2,1,1,.github/workflows/docker-connector-base-image-...,modified,1.0,1.0,2.0,"@@ -367,5 +367,5 @@ jobs:\n image: ""..."
679,b6cf5e5711ce1e154575cec8aa91f69f1634c049,2932761138,lordsarcastic,web-flow,Merge branch 'master' into devin/ext-398-aws-s...,69243,39624,29619,.github/workflows/build-image.yaml,modified,5.0,0.0,5.0,"@@ -44,3 +44,8 @@ jobs:\n if: en..."
680,b6cf5e5711ce1e154575cec8aa91f69f1634c049,2932761138,lordsarcastic,web-flow,Merge branch 'master' into devin/ext-398-aws-s...,69243,39624,29619,.github/workflows/build-images.yaml,removed,0.0,29.0,29.0,"@@ -1,29 +0,0 @@\n-name: '[Release] Build serv..."
681,b6cf5e5711ce1e154575cec8aa91f69f1634c049,2932761138,lordsarcastic,web-flow,Merge branch 'master' into devin/ext-398-aws-s...,69243,39624,29619,.github/workflows/deploy.yaml,modified,9.0,10.0,19.0,"@@ -32,18 +32,17 @@ jobs:\n steps:\n ..."
682,b6cf5e5711ce1e154575cec8aa91f69f1634c049,2932761138,lordsarcastic,web-flow,Merge branch 'master' into devin/ext-398-aws-s...,69243,39624,29619,.github/workflows/push-container.yaml,removed,0.0,42.0,42.0,"@@ -1,42 +0,0 @@\n-name: Push container\n-\n-o..."


In [6]:
ci_commits["filename"].value_counts().head(20)

filename
.github/workflows/ci.yml                                   637
.github/workflows/test.yml                                 185
.github/workflows/copilot-setup-steps.yml                  172
.github/workflows/build.yml                                134
.github/workflows/docs.yml                                 126
.github/workflows/release.yml                              110
.circleci/config.yml                                        78
.github/workflows/tests.yml                                 75
.github/workflows/claude.yml                                69
.github/workflows/main.yml                                  64
.github/workflows/build-and-test.yml                        57
.github/workflows/e2e.yml                                   42
.github/workflows/size-check.yml                            42
.github/workflows/python_pytest.yml                         41
.github/workflows/lint.yml                                  38
.github/workflows/ppa-publish.yml             

In [7]:
sample = ci_commits.sample(10, random_state=42)

for idx, row in sample.iterrows():
    
    print("="*80)
    print(f"PR ID: {row['pr_id']}")
    print(f"FILE: {row['filename']}")
    print(f"STATUS: {row['status']}")
    print(f"MESSAGE: {row['message']}")
    print("-"*80)
    
    print(row['patch'])
    print("\n")

PR ID: 3157140778
FILE: .github/workflows/build_docs.yml
STATUS: modified
MESSAGE: Disable publishing remote config unless repo has env var set, to differentate from docs repo
--------------------------------------------------------------------------------
@@ -51,6 +51,7 @@ jobs:
 
   # Deployment job
   deploy:
+    if: ${{ vars.KILN_CONFIG_REPO != 'true' }}
     environment:
       name: github-pages
       url: ${{ steps.deployment.outputs.page_url }}


PR ID: 3057698280
FILE: .github/workflows/ci-pythnet-sdk.yml
STATUS: modified
MESSAGE: Merge branch 'main' of github.com:pyth-network/pyth-crosschain into devin/1747071611-reset-pricelastupdatedat-on-subscription-update
--------------------------------------------------------------------------------
@@ -20,8 +20,8 @@ jobs:
           workspaces: "pythnet/pythnet_sdk -> target"
       - uses: actions-rs/toolchain@v1
         with:
-          profile: minimal
           toolchain: 1.82.0
+          components: rustfmt, clippy
         

In [8]:
pull_request.columns

Index(['id', 'number', 'title', 'user', 'user_id', 'state', 'created_at',
       'closed_at', 'merged_at', 'repo_url', 'repo_id', 'html_url', 'body',
       'agent'],
      dtype='object')

In [9]:
pr_status = pull_request[
    [
        "id",
        "state",
        "created_at",
        "closed_at",
        "merged_at",
        "agent",
        "repo_url",
        "html_url"
    ]
].copy()

In [10]:
pr_status.rename(
    columns={"id": "pr_id"},
    inplace=True
)

In [11]:
ci_commits = ci_commits.merge(
    pr_status,
    on="pr_id",
    how="left"
)

In [12]:
ci_commits["pr_result"] = np.where(
    ci_commits["merged_at"].notna(),
    "accepted",
    np.where(
        ci_commits["closed_at"].notna(),
        "rejected",
        "open"
    )
)

In [13]:
ci_commits["pr_result"].value_counts()

pr_result
accepted    4822
rejected    1906
open        1132
Name: count, dtype: int64

In [14]:
ci_commits_final = ci_commits[
    ci_commits["pr_result"].isin(["accepted", "rejected"])
].copy()

print(ci_commits_final["pr_result"].value_counts())

pr_result
accepted    4822
rejected    1906
Name: count, dtype: int64


In [15]:
sample = ci_commits.sample(20, random_state=42)

for idx, row in sample.iterrows():

    print("="*100)
    print(f"PR ID: {row['pr_id']}")
    print(f"RESULT: {row['pr_result']}")
    print(f"AGENT: {row['agent']}")
    print(f"FILE: {row['filename']}")
    print(f"MESSAGE: {row['message']}")
    print("-"*100)

    print(row["patch"])
    print("\n\n")

PR ID: 3157140778
RESULT: accepted
AGENT: OpenAI_Codex
FILE: .github/workflows/build_docs.yml
MESSAGE: Disable publishing remote config unless repo has env var set, to differentate from docs repo
----------------------------------------------------------------------------------------------------
@@ -51,6 +51,7 @@ jobs:
 
   # Deployment job
   deploy:
+    if: ${{ vars.KILN_CONFIG_REPO != 'true' }}
     environment:
       name: github-pages
       url: ${{ steps.deployment.outputs.page_url }}



PR ID: 3057698280
RESULT: accepted
AGENT: Devin
FILE: .github/workflows/ci-pythnet-sdk.yml
MESSAGE: Merge branch 'main' of github.com:pyth-network/pyth-crosschain into devin/1747071611-reset-pricelastupdatedat-on-subscription-update
----------------------------------------------------------------------------------------------------
@@ -20,8 +20,8 @@ jobs:
           workspaces: "pythnet/pythnet_sdk -> target"
       - uses: actions-rs/toolchain@v1
         with:
-          profile: minimal
   

In [16]:
sample = ci_commits_final.sample(5)

sample[[
    "pr_id",
    "html_url",
    "message",
    "filename"
]]

,pr_id,html_url,message,filename
3913,3222841302,https://github.com/docker/hello-genai/pull/8,Remove .env file and clean up docker-compose c...,.github/workflows/smoke-test.yml
7280,3070957415,https://github.com/pardeike/Harmony/pull/676,Refine CI labels,.github/workflows/test.yml
2988,3164310659,https://github.com/microsoft/promptpex/pull/179,Merge latest dev branch\n\n- Merged dev branch...,.github/workflows/genai-pr.yml
4524,3127781810,https://github.com/DaveSkender/Stock.Indicator...,Update test-indicators.yml\n\nSigned-off-by: D...,.github/workflows/test-indicators.yml
238,3096353676,https://github.com/ensdomains/ens-app-v3/pull/...,Merge branch 'main' into devin/1748417263-upda...,.github/workflows/pages-deployment.yaml


In [17]:
pd.set_option('display.max_colwidth', None)

In [18]:
sample[[
    "pr_id",
    "html_url",
    "message",
    "filename"
]].head()

,pr_id,html_url,message,filename
3913,3222841302,https://github.com/docker/hello-genai/pull/8,Remove .env file and clean up docker-compose configurations\n\nCo-authored-by: kiview <5088104+kiview@users.noreply.github.com>,.github/workflows/smoke-test.yml
7280,3070957415,https://github.com/pardeike/Harmony/pull/676,Refine CI labels,.github/workflows/test.yml
2988,3164310659,https://github.com/microsoft/promptpex/pull/179,"Merge latest dev branch\n\n- Merged dev branch containing Docker configurations, CI/CD improvements, and enhanced eval model support\n- Resolved conflict in reports.mts by applying NaN fix to new multi-eval-model structure\n- Preserved original NaN handling fix while adopting new evalModels functionality",.github/workflows/genai-pr.yml
4524,3127781810,https://github.com/DaveSkender/Stock.Indicators/pull/1364,Update test-indicators.yml\n\nSigned-off-by: Dave Skender <8432125+DaveSkender@users.noreply.github.com>,.github/workflows/test-indicators.yml
238,3096353676,https://github.com/ensdomains/ens-app-v3/pull/1016,Merge branch 'main' into devin/1748417263-update-eth-wallet-message,.github/workflows/pages-deployment.yaml


In [19]:
def classify_workflow(filename):

    filename = filename.lower()

    if "test" in filename or "pytest" in filename or "e2e" in filename:
        return "testing"

    elif "build" in filename:
        return "build"

    elif "release" in filename or "publish" in filename:
        return "release"

    elif "lint" in filename:
        return "quality"

    elif "docs" in filename:
        return "documentation"

    elif "claude" in filename or "copilot" in filename:
        return "agent_automation"

    elif "ci" in filename:
        return "ci_core"

    else:
        return "other"


ci_commits_final["workflow_category"] = (
    ci_commits_final["filename"]
    .apply(classify_workflow)
)

In [20]:
ci_commits_final["created_at"] = pd.to_datetime(
    ci_commits_final["created_at"]
)

ci_commits_final["closed_at"] = pd.to_datetime(
    ci_commits_final["closed_at"]
)

ci_commits_final["resolution_hours"] = (
    ci_commits_final["closed_at"]
    - ci_commits_final["created_at"]
).dt.total_seconds() / 3600

In [21]:
sample = ci_commits_final.sample(20, random_state=42)

sample_view = sample[
    [
        "pr_id",
        "html_url",
        "agent",
        "filename",
        "workflow_category",
        "pr_result",
        "resolution_hours",
        "message",
        "patch"
    ]
]

pd.set_option('display.max_colwidth', None)

display(sample_view)

pr_id                                                    html_url  \
5884  3145240427          https://github.com/antelle/argon2-browser/pull/101   
142   3136450775                    https://github.com/novuhq/novu/pull/8498   
5583  3276266000                 https://github.com/calganaygun/MDcat/pull/1   
5602  3245540159  https://github.com/MontrealAI/AGI-Alpha-Agent-v0/pull/3567   
3184  3084107311             https://github.com/microsoft/typespec/pull/7439   
1679  3057663648             https://github.com/airbytehq/airbyte/pull/60225   
454   2989797052                https://github.com/antiwork/flexile/pull/110   
1566  2798936903          https://github.com/appsmithorg/appsmith/pull/38772   
3941  3188535011          https://github.com/microsoft/genaiscript/pull/1694   
2479  3226808853          https://github.com/microsoft/genaiscript/pull/1742   
5428  3224473633  https://github.com/MontrealAI/AGI-Alpha-Agent-v0/pull/3232   
1148  3023352746     https://github.com/altive/flutter_app_template/pull/564   
2529  3125850603        https://github.com/microsoft/typescript-go/pull/1086   
3679  3179789638         https://github.com/valkey-io/valkey-glide/pull/4264   
6450  3219422853  https://github.com/MontrealAI/AGI-Alpha-Agent-v0/pull/3182   
5470  3070077638           https://github.com/joshuafuller/ATAK-Maps/pull/38   
6933  3129247657              https://github.com/netbirdio/netbird/pull/3944   
7562  3263487993  https://github.com/MontrealAI/AGI-Alpha-Agent-v0/pull/3756   
572   3177562468                   https://github.com/liam-hq/liam/pull/2220   
5204  3236103019         https://github.com/getsentry/sentry-cocoa/pull/5644   

             agent                                       filename  \
5884  OpenAI_Codex           .github/workflows/docker-publish.yml   
142          Devin       .github/workflows/deployment-summary.yml   
5583  OpenAI_Codex                     .github/workflows/main.yml   
5602  OpenAI_Codex                       .github/workflows/ci.yml   
3184       Copilot                    .github/workflows/README.md   
1679         Devin  .github/workflows/auto_merge_notification.yml   
454          Devin                       .github/workflows/ci.yml   
1566         Devin             .github/workflows/client-build.yml   
3941       Copilot                .github/workflows/npm-check.yml   
2479       Copilot                 .github/workflows/cli-test.yml   
5428  OpenAI_Codex           .github/workflows/build-and-test.yml   
1148         Devin   .github/workflows/flutter-app-code-check.yml   
2529       Copilot                       .github/workflows/ci.yml   
3679       Copilot                       .github/workflows/go.yml   
6450  OpenAI_Codex                     .github/workflows/docs.yml   
5470  OpenAI_Codex             .github/workflows/super-linter.yml   
6933  OpenAI_Codex  .github/workflows/mobile-build-validation.yml   
7562  OpenAI_Codex                       .github/workflows/ci.yml   
572          Devin                  .github/workflows/release.yml   
5204        Cursor                    .github/workflows/build.yml   

     workflow_category pr_result  resolution_hours  \
5884           release  rejected          0.082222   
142              other  accepted          1.906944   
5583             other  accepted          0.793611   
5602           ci_core  accepted          0.003056   
3184             other  accepted         24.655556   
1679             other  accepted        191.360278   
454            ci_core  accepted        166.021389   
1566             build  rejected        725.623611   
3941             other  rejected         46.679167   
2479           testing  accepted          5.144444   
5428           testing  accepted          0.003333   
1148             other  accepted         30.741111   
2529           ci_core  rejected        434.927778   
3679             other  rejected        160.528611   
6450     documentation  accepted          0.002778   
5470           q

In [22]:
# Crear columna vacía para etiquetado manual
ci_commits_final["modification_type"] = ""

# Seleccionar columnas relevantes
export_df = ci_commits_final[
    [
        "pr_id",
        "html_url",
        "agent",
        "filename",
        "workflow_category",
        "pr_result",
        "resolution_hours",
        "message",
        "patch",
        "modification_type"
    ]
].copy()

# Exportar CSV
export_df.to_csv(
    "ci_cd_pr_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV generado correctamente: ci_cd_pr_analysis.csv")
print(f"Total de casos exportados: {len(export_df)}")

CSV generado correctamente: ci_cd_pr_analysis.csv
Total de casos exportados: 6728
